## Plot fraction of individuals with low neutralization titers by strain

In [31]:
# Import packages
import os
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gmean
from IPython.display import IFrame


# Ignore error message from Altair about large dataframes
_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
               ]

In [32]:
# Define inputs
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok = True)

# Define titers
titers = pd.read_csv(
    '../../../results/aggregated_titers/titers_EllebedyVaccineCohort.csv'
)

titers = titers.assign(
    vaccine = lambda x: x['serum'].str.split('_').str[0],
    participant = lambda x: x['serum'].str.split('_').str[1],
    timepoint = lambda x: x['serum'].str.split('_').str[2],
    subtype = lambda x: x['virus'].str.split('_').str[-1]
)

serum_to_inlcude = [
  #plate1
  'Fluarix_397-026_d0',
  'Fluarix_397-026_d29',
  'Fluarix_397-026_d181',
  'mRNA_397-031_d0',
  'mRNA_397-031_d29',
  'mRNA_397-031_d181',
  'mRNA_397-010_d0',
  'mRNA_397-010_d29',
  'mRNA_397-010_d12',
    #plate4
  'Fluarix_397-032_d0',
  'Fluarix_397-032_d29',
  'Fluarix_397-032_d181',
  'Fluarix_397-022_d0',
  'Fluarix_397-022_d29',
  'Fluarix_397-022_d181',
  'mRNA_397-017_d0',
  'mRNA_397-017_d29',
  'mRNA_397-017_d181',
  'mRNA_397-021_d0',
  'mRNA_397-021_d29',
  'mRNA_397-021_d181',
  #plate5
  'Fluarix_397-020_d0',
  'Fluarix_397-020_d29',
  'Fluarix_397-020_d181',
  'Fluarix_397-018_d0',
  'Fluarix_397-018_d29',
  'Fluarix_397-018_d181',
  'mRNA_397-008_d0',
  'mRNA_397-008_d29',
  'mRNA_397-008_d181',
  'mRNA_397-028_d0',
  'mRNA_397-028_d29',
  'mRNA_397-028_d181'    
]

# Uncomment to pare down sera to above list
titers = titers[titers['serum'].isin(serum_to_inlcude)]
titers.serum.unique()

array(['Fluarix_397-018_d0', 'Fluarix_397-018_d29', 'Fluarix_397-020_d0',
       'Fluarix_397-020_d181', 'Fluarix_397-020_d29',
       'Fluarix_397-022_d0', 'Fluarix_397-022_d181',
       'Fluarix_397-022_d29', 'Fluarix_397-026_d0',
       'Fluarix_397-026_d181', 'Fluarix_397-026_d29',
       'Fluarix_397-032_d0', 'Fluarix_397-032_d181',
       'Fluarix_397-032_d29', 'mRNA_397-008_d0', 'mRNA_397-008_d181',
       'mRNA_397-008_d29', 'mRNA_397-010_d0', 'mRNA_397-010_d29',
       'mRNA_397-017_d0', 'mRNA_397-017_d181', 'mRNA_397-017_d29',
       'mRNA_397-021_d0', 'mRNA_397-021_d181', 'mRNA_397-021_d29',
       'mRNA_397-028_d0', 'mRNA_397-028_d181', 'mRNA_397-028_d29',
       'mRNA_397-031_d0', 'mRNA_397-031_d181', 'mRNA_397-031_d29'],
      dtype=object)

In [33]:
# Define virus order
viral_plot_order = pd.read_csv('../../../data/H3_H1_ellebedy_library_strain_order.csv')
virus_order = [v for v in viral_plot_order.strain]

## Plot neutralization titers

In [34]:
def plot_titers_vaccination_cohorts(data, sort_order, _range = [30, 20000], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('timepoint', sort=['d0', 'd29']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y('titer', 
                          scale =alt.Scale(type='log',domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title="neutralization titer")),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y('median(titer)'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .facet(row = alt.Row('vaccine:N', sort=sort_order),
                      config = alt.Config(legend = alt.LegendConfig(titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                                                    strokeColor='gray',padding=10,cornerRadius=10,
                                                    labelLimit = 1000 # Let legend labels be as long as they want
                                                     )))
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

In [35]:
for subtype in titers.subtype.unique():
    data = titers[titers['subtype'].isin([subtype])]

    plot = plot_titers_vaccination_cohorts(data, sort_order = ['Fluarix', 'mRNA-1010'], 
                                           _range=[30, 18000], title = f'{subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_pre-post_titers.pdf')
    plot.save(outfile, dpi = 600)


In [36]:
IFrame(os.path.join(resultsdir, f'H3N2_pre-post_titers.pdf'), width=700, height=500)

In [37]:
IFrame(os.path.join(resultsdir, f'H1N1_pre-post_titers.pdf'), width=700, height=500)

# Plot log fold-change titers for each group from day 0 to day 29 and day 0 to day 121/181

In [38]:
# Pivot the table to have timepoints as columns
pivot_titers = titers.pivot_table(
    index=['participant', 'vaccine', 'virus', 'subtype', ],
    columns='timepoint',
    values='titer'
)

# Calculate fold-changes
pivot_titers['fold_change_d29'] = pivot_titers['d29'] / pivot_titers['d0']
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121'] = pivot_titers['d121'] / pivot_titers['d0']
pivot_titers['fold_change_d181'] = pivot_titers['d181'] / pivot_titers['d0']

# There's inconsistency between the last date being d121 or d181, so making a column that has both
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121_or_d181'] = pivot_titers.get('d121', pd.NA).combine_first(pivot_titers.get('d181', pd.NA)) / pivot_titers['d0']

# Compute log2 fold changes
pivot_titers['log2_fc_d29'] = np.log2(pivot_titers['fold_change_d29'])
pivot_titers['log2_fc_d181'] = np.log2(pivot_titers['fold_change_d181'])
if 'd121' in pivot_titers.columns:
    pivot_titers['log2_fc_d121'] = np.log2(pivot_titers['fold_change_d121'])
    pivot_titers['log2_fc_d121_or_d181'] = np.log2(pivot_titers['fold_change_d121_or_d181'])

pivot_titers = pivot_titers.reset_index()
pivot_titers

timepoint,participant,vaccine,virus,subtype,d0,d181,d29,fold_change_d29,fold_change_d181,log2_fc_d29,log2_fc_d181
0,397-008,mRNA,A/AbuDhabi/6753/2023_H3N2,H3N2,140.9,123.4,274.5,1.948190,0.875798,0.962135,-0.191329
1,397-008,mRNA,A/Arizona/IVYG43Q65T3/2023_H1N1,H1N1,111.6,121.5,160.8,1.440860,1.088710,0.526930,0.122619
2,397-008,mRNA,A/Bangkok/P3599/2023_H3N2,H3N2,196.4,191.1,181.3,0.923116,0.973014,-0.115416,-0.039467
3,397-008,mRNA,A/Bangkok/P3755/2023_H3N2,H3N2,157.7,128.3,288.7,1.830691,0.813570,0.872388,-0.297661
4,397-008,mRNA,A/Bhutan/0006/2023_H3N2,H3N2,123.5,184.3,208.5,1.688259,1.492308,0.755536,0.577545
...,...,...,...,...,...,...,...,...,...,...,...
1062,397-032,Fluarix,A/Victoria/4897/2022_R142K_H1N1,H1N1,1419.0,1000.0,1419.0,1.000000,0.704722,0.000000,-0.504875
1063,397-032,Fluarix,A/Wisconsin/27/2023_H3N2,H3N2,190.4,588.9,3188.0,16.743697,3.092962,4.065546,1.628989
1064,397-032,Fluarix,A/Wisconsin/588/2019_H1N1,H1N1,752.0,645.8,824.1,1.095878,0.858777,0.132087,-0.219645
1065,397-032,Fluarix,A/Wisconsin/67/2022_H1N1,H1N1,113.5,118.3,282.0,2.484581,1.042291,1.313003,0.059758


In [39]:
def plot_FC_vaccination_cohorts(data, y_axis, sort_order, _range = [0, 1], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('vaccine', sort=['mRNA-1010', 'Fluarix']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y(f'{y_axis}', 
                          scale =alt.Scale(domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title=["log FC titer", "from pre-vaccination"])),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y(f'median({y_axis})'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

## Log FC in neutralization titers at day 29

In [40]:
for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts(data, 
                                           y_axis = 'log2_fc_d29',
                                           sort_order = ['mRNA-1010', 'Fluarix'], 
                                           _range=[-0.6, 6], title = f'Log FC day 29 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day29_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [41]:
IFrame(os.path.join(resultsdir, 'H3N2_day29_pre-post_fold-change.pdf'), width=700, height=500)

In [42]:
IFrame(os.path.join(resultsdir, 'H1N1_day29_pre-post_fold-change.pdf'), width=700, height=500)

## Log FC in neutralization titers at day 121 or 181

In [43]:
if 'd121' in pivot_titers.columns:
    y_axis = 'log2_fc_d121_or_d181'
else:
    y_axis = 'log2_fc_d181'

for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts(data, 
                                           y_axis = y_axis,
                                           sort_order = ['mRNA-1010', 'Fluarix', ], 
                                           _range=[-1.5, 4.5], title = f'Log FC day 121/181 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day121-181_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [44]:
IFrame(os.path.join(resultsdir, 'H3N2_day121-181_pre-post_fold-change.pdf'), width=700, height=500)

In [45]:
IFrame(os.path.join(resultsdir, 'H1N1_day121-181_pre-post_fold-change.pdf'), width=700, height=500)